<a href="https://colab.research.google.com/github/Prajwaln08/SQLSmith/blob/main/notebooks/03_Spider_Official_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQLSmith — 03 | Spider Official Test-Suite Evaluation

**Milestone A · Qwen2.5-Coder-1.5B-Instruct · Spider 1.0 · CPU-only**

This notebook **reuses the 50 saved model predictions** from Notebook 01. It does not load the LLM or regenerate SQL. It validates sample alignment and runs the [official Spider test-suite evaluator](https://github.com/taoyds/test-suite-sql-eval) using the **augmented test-suite database package**, not just the original Spider SQLite databases.

**Comparability:** original toy evaluator 24/50 (48%); diagnostic replay 26/50 (52%); official **first-50 subset** result pending. Neither previous number is a full Spider benchmark score. The original CSV is never overwritten.


## 1. Environment and saved predictions

In [1]:
%pip -q install pandas numpy huggingface_hub gdown sqlparse nltk tqdm

In [2]:
from google.colab import drive
from pathlib import Path
import hashlib, json, sys, subprocess, re, os
import pandas as pd

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/Prajwal/Projects/SQLSmith')
RESULTS_DIR = PROJECT_DIR / 'results'
EVAL_DIR = PROJECT_DIR / 'evaluation' / 'spider_50'
EVAL_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_FILE = RESULTS_DIR / 'spider_1_5b_baseline_50.csv'
assert BASELINE_FILE.is_file(), f'Missing Notebook 01 CSV: {BASELINE_FILE}'
def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()
source_hash = sha256_file(BASELINE_FILE)
print('Input:', BASELINE_FILE)
print('SHA256:', source_hash)
print('Outputs:', EVAL_DIR)

Mounted at /content/drive
Input: /content/drive/MyDrive/Prajwal/Projects/SQLSmith/results/spider_1_5b_baseline_50.csv
SHA256: 3d714667a99969fb16090359564da301d1b478eeba2bc1c376760d96fc953f3e
Outputs: /content/drive/MyDrive/Prajwal/Projects/SQLSmith/evaluation/spider_50


In [3]:
df = pd.read_csv(BASELINE_FILE, keep_default_na=False)
required = {'index','db_id','question','gold_sql','predicted_sql','correct'}
assert required.issubset(df.columns), f'Missing columns: {required - set(df.columns)}'
df['index'] = pd.to_numeric(df['index'], errors='raise').astype(int)
df = df.sort_values('index').reset_index(drop=True)
assert len(df)==50 and df['index'].tolist()==list(range(1,51)), 'Expected original first 50 Spider dev questions'
assert df['index'].is_unique
assert df[['db_id','question','gold_sql']].apply(lambda s: s.astype(str).str.strip().ne('').all()).all()
original_correct = sum(str(v).lower() in ('true','1') for v in df['correct'])
print(f'Frozen rows: {len(df)} | Original provisional passes: {original_correct}/{len(df)}')
display(df[['index','db_id','question','predicted_sql']].head(5))

Frozen rows: 50 | Original provisional passes: 24/50


,index,db_id,question,predicted_sql
0,1,concert_singer,How many singers do we have?,SELECT count(*) FROM singer
1,2,concert_singer,What is the total number of singers?,SELECT count(*) FROM singer
2,3,concert_singer,"Show name, country, age for all singers ordere...","SELECT Name, Country, Age FROM singer ORDER BY..."
3,4,concert_singer,"What are the names, countries, and ages for ev...","SELECT name, country, age FROM singer ORDER BY..."
4,5,concert_singer,"What is the average, minimum, and maximum age ...","SELECT AVG(Age) , MIN(Age) , MAX(Age) FROM s..."


## 2. Confirm original Spider question alignment

The public Spider package provides the original development questions and schema metadata. The test-suite package downloaded later is **different** and is required for official test-suite scoring.

In [4]:
from huggingface_hub import hf_hub_download
import zipfile

SPIDER_DIR = Path('/content/sqlsmith_data/spider_data')
archive = hf_hub_download(repo_id='HAL-9001/spider-databases', repo_type='dataset', filename='spider_data.zip')
expected_hash = '00636695dabed6b5f4b8328a16b13e069a2f16591d5efcce57660669c85b121b'
assert sha256_file(archive) == expected_hash, 'Spider data ZIP checksum differs: inspect source/version'
if not (SPIDER_DIR/'dev.json').exists() or not (SPIDER_DIR/'tables.json').exists():
    extract_root = Path('/content/sqlsmith_data').resolve()
    extract_root.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for m in z.infolist():
            dest=(extract_root/m.filename).resolve()
            assert dest.is_relative_to(extract_root), f'Unsafe path in ZIP: {m.filename}'
        z.extractall(extract_root)
with open(SPIDER_DIR/'dev.json', encoding='utf-8') as f:
    dev=json.load(f)
TABLES_FILE=SPIDER_DIR/'tables.json'
assert len(dev)>=50
for _, r in df.iterrows():
    orig=dev[int(r['index'])-1]
    assert str(r['db_id'])==orig['db_id'], f'DB mismatch at row {r["index"]}'
    assert str(r['question']).strip()==orig['question'].strip(), f'Question mismatch at row {r["index"]}'
    assert ' '.join(str(r['gold_sql']).split())==' '.join(orig['query'].split()), f'Gold mismatch at row {r["index"]}'
print(f'PASS: all 50 rows align with Spider dev; total available dev questions: {len(dev)}')

spider_data.zip: reconstructing file:   0%|          |  0.00B /  206MB            

spider_data.zip: downloading bytes:           |  0.00B            

PASS: all 50 rows align with Spider dev; total available dev questions: 1034


## 3. Official evaluator code and exact revision

In [5]:
EVALUATOR_DIR=Path('/content/test-suite-sql-eval')
if not (EVALUATOR_DIR/'evaluation.py').is_file():
    subprocess.run(['git','clone','--depth','1','https://github.com/taoyds/test-suite-sql-eval.git',str(EVALUATOR_DIR)],check=True)
evaluator_commit=subprocess.check_output(['git','-C',str(EVALUATOR_DIR),'rev-parse','HEAD'],text=True).strip()
assert (EVALUATOR_DIR/'exec_eval.py').is_file()
print('Official evaluator commit:', evaluator_commit)
print('Python:', sys.version.split()[0])

Official evaluator commit: e97acc546ecbee8fa27fa8dbf025ef61493a876c
Python: 3.13.18


## 4. Official augmented test-suite databases

The evaluator README links an archive at [Google Drive](https://drive.google.com/file/d/1mkCx2GOFIqNesD4y8TDAO1yX1QZORP5w/view). Auto-download is attempted. If it fails, download the archive manually and place it at `/content/testsuitedatabases.zip`, then rerun this cell. **Do not substitute the normal Spider database ZIP.**

In [7]:

from pathlib import Path
import zipfile
import shutil

# Persistent Google Drive location
DRIVE_ARCHIVE = Path(
    "/content/drive/MyDrive/Prajwal/Projects/"
    "SQLSmith/data/testsuitedatabases.zip"
)

# Temporary Colab locations
TEST_SUITE_ARCHIVE = Path(
    "/content/testsuitedatabases.zip"
)

SUITE_EXTRACT = Path(
    "/content/sqlsmith_test_suites"
)

SUITE_EXTRACT.mkdir(parents=True, exist_ok=True)

# Reuse existing archive if available
if DRIVE_ARCHIVE.is_file():
    print("Found ZIP in Google Drive.")
    shutil.copy2(DRIVE_ARCHIVE, TEST_SUITE_ARCHIVE)

elif TEST_SUITE_ARCHIVE.is_file():
    print("Found ZIP in Colab.")

else:
    raise FileNotFoundError(
        "Test-suite ZIP unavailable. Download the "
        "official archive manually and save it at: "
        + str(DRIVE_ARCHIVE)
    )

# Verify ZIP
assert zipfile.is_zipfile(TEST_SUITE_ARCHIVE), (
    "File is not a valid ZIP archive."
)

print(
    "Archive size:",
    round(TEST_SUITE_ARCHIVE.stat().st_size / 1e6, 2),
    "MB"
)

# Safe extraction
with zipfile.ZipFile(TEST_SUITE_ARCHIVE) as z:
    root = SUITE_EXTRACT.resolve()

    for member in z.infolist():
        destination = (root / member.filename).resolve()
        assert destination.is_relative_to(root), (
            f"Unsafe ZIP entry: {member.filename}"
        )

    z.extractall(root)

print("Extraction complete:", SUITE_EXTRACT)


Found ZIP in Google Drive.
Archive size: 1269.46 MB
Extraction complete: /content/sqlsmith_test_suites


In [8]:
required_dbids=set(df['db_id'].astype(str))
# Official evaluator expects <root>/<db_id>/<db_id>.sqlite and scans every *.sqlite in that subfolder.
possible=[]
for anchor_id in required_dbids:
    for base_file in SUITE_EXTRACT.rglob(f'{anchor_id}.sqlite'):
        if base_file.parent.name!=anchor_id:
            continue
        root=base_file.parent.parent
        if all((root/d/(d+'.sqlite')).is_file() for d in required_dbids):
            if root not in possible:
                possible.append(root)
assert possible, 'Could not locate official suite root with required <db_id>/<db_id>.sqlite files'
# Prefer candidate with most .sqlite files in target schema directories.
def count_suites(root):
    return sum(len(list((root/d).glob('*.sqlite'))) for d in required_dbids)
SUITE_DB_ROOT=max(possible,key=count_suites)
suite_counts={d:len(list((SUITE_DB_ROOT/d).glob('*.sqlite'))) for d in sorted(required_dbids)}
assert min(suite_counts.values())>=1
assert max(suite_counts.values())>1, ('No augmented DB instances found. This appears to be original Spider databases, not test suites.')
print('Selected suite root:', SUITE_DB_ROOT)
print('Number of .sqlite files for each required schema:',suite_counts)
print('PASS: extra database instances detected')

Selected suite root: /content/sqlsmith_test_suites/database
Number of .sqlite files for each required schema: {'concert_singer': 32, 'pets_1': 27}
PASS: extra database instances detected


## 5. Build gold and prediction files

The official tool expects **one line of gold SQL + tab + db ID** per example, and one line of predicted SQL per example. To avoid silently changing SQL literals, cells *reject* multiline/tabbed predictions instead of rewriting them. Empty predictions also stop this run for explicit review, rather than silently replacing them.

In [9]:
GOLD_FILE=EVAL_DIR/'spider_50_gold.txt'
PRED_FILE=EVAL_DIR/'spider_50_predictions.txt'
def one_line_sql(value,row_id,field):
    value=str(value)
    assert value.strip(), f'Empty {field} at question {row_id}; investigate before official scoring'
    assert '\n' not in value and '\r' not in value and '\t' not in value, f'Multiline/tab SQL at question {row_id} ({field})'
    return value.strip()
gold_lines=[]; pred_lines=[]
for _,r in df.iterrows():
    idx=int(r['index'])
    gold_lines.append(one_line_sql(r['gold_sql'],idx,'gold_sql')+'\t'+str(r['db_id']))
    pred_lines.append(one_line_sql(r['predicted_sql'],idx,'predicted_sql'))
GOLD_FILE.write_text('\n'.join(gold_lines)+'\n',encoding='utf-8')
PRED_FILE.write_text('\n'.join(pred_lines)+'\n',encoding='utf-8')
assert len(GOLD_FILE.read_text(encoding='utf-8').splitlines())==50
assert len(PRED_FILE.read_text(encoding='utf-8').splitlines())==50
print('PASS: 50 gold/prediction pairs written without SQL rewriting')
print('Gold sha256:',sha256_file(GOLD_FILE))
print('Pred sha256:',sha256_file(PRED_FILE))

PASS: 50 gold/prediction pairs written without SQL rewriting
Gold sha256: 2523871ac4fa312076a17448302d1778c254e5b072bb0e3cc509dd79e7eeb930
Pred sha256: 8237e4bd4a1ef2ca01dea87c09743cfa0fc63a714446de80c070b0f1f5a8aced


## 6. Run official test-suite execution scorer

`--etype exec`: execution-based scoring. We **do not** use `--plug_value` (no gold-value oracle). The official tool has its own evaluation conventions (including default distinct handling), so this score cannot be compared as if it were the toy exact-row comparison.

In [11]:

import nltk

# Download tokenizer dependencies required by Spider evaluator
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# Verify the required resource exists
resource = nltk.data.find(
    "tokenizers/punkt_tab/english/"
)

print("NLTK punkt_tab found:", resource)
print("NLTK version:", nltk.__version__)
print("Ready to run official Spider evaluation.")


NLTK punkt_tab found: /root/nltk_data/tokenizers/punkt_tab/english
NLTK version: 3.9.1
Ready to run official Spider evaluation.


In [12]:
import time
LOG_FILE=EVAL_DIR/'spider_50_official_evaluation.log'
command=[sys.executable,str(EVALUATOR_DIR/'evaluation.py'), '--gold',str(GOLD_FILE),'--pred',str(PRED_FILE), '--etype','exec', '--db',str(SUITE_DB_ROOT), '--table',str(TABLES_FILE)]
print('Running:', ' '.join(command))
start=time.monotonic()
try:
    proc=subprocess.run(command,cwd=str(EVALUATOR_DIR),capture_output=True,text=True,timeout=1200)
except subprocess.TimeoutExpired as e:
    LOG_FILE.write_text('Evaluator timed out after 1200 seconds.\n'+str(e),encoding='utf-8')
    raise
elapsed=time.monotonic()-start
LOG_FILE.write_text('COMMAND: '+' '.join(command)+'\n\nSTDOUT:\n'+proc.stdout+'\nSTDERR:\n'+proc.stderr,encoding='utf-8')
print('Exit code:',proc.returncode,'Elapsed seconds:',round(elapsed,1))
print('\n--- STDOUT (tail) ---\n',proc.stdout[-7000:])
if proc.stderr:
    print('\n--- STDERR (tail) ---\n',proc.stderr[-3500:])
assert proc.returncode==0, f'Official evaluation failed; see {LOG_FILE}'
assert 'EXECUTION ACCURACY' in proc.stdout, f'Execution accuracy table missing; see {LOG_FILE}'
print('PASS: official evaluator exited successfully; log saved to Drive')

Running: /usr/bin/python3 /content/test-suite-sql-eval/evaluation.py --gold /content/drive/MyDrive/Prajwal/Projects/SQLSmith/evaluation/spider_50/spider_50_gold.txt --pred /content/drive/MyDrive/Prajwal/Projects/SQLSmith/evaluation/spider_50/spider_50_predictions.txt --etype exec --db /content/sqlsmith_test_suites/database --table /content/sqlsmith_data/spider_data/tables.json
Exit code: 0 Elapsed seconds: 4.3

--- STDOUT (tail) ---
                      easy                 medium               hard                 extra                all                 
count                6                    27                   13                   4                    50                  
=====================   EXECUTION ACCURACY     =====================
execution            0.667                0.407                0.308                0.500                0.420               

PASS: official evaluator exited successfully; log saved to Drive


## 7. Parse the official subset score and save reproducible metadata

In [13]:
# For Spider, evaluator prints levels: easy, medium, hard, extra, all.
lines=proc.stdout.splitlines()
header_idx=next((i for i,line in enumerate(lines) if 'EXECUTION ACCURACY' in line),None)
assert header_idx is not None
score_line=next((line for line in lines[header_idx+1:] if re.match(r'^\s*execution\s+',line)),None)
assert score_line is not None, 'Could not parse execution accuracy row; inspect log'
parts=score_line.strip().split()
assert parts[0]=='execution' and len(parts)>=6, f'Unexpected score row: {score_line}'
score_by_difficulty=dict(zip(['easy','medium','hard','extra','all'],[float(x) for x in parts[1:6]]))
assert all(0<=v<=1 for v in score_by_difficulty.values())
score=score_by_difficulty['all']
print('Official Spider test-suite FIRST-50 SUBSET accuracy:',f'{score*100:.1f}%')
print('Difficulty scores:',score_by_difficulty)
print('Previously observed: original 48%, diagnostic replay 52% (different protocol)')

Official Spider test-suite FIRST-50 SUBSET accuracy: 42.0%
Difficulty scores: {'easy': 0.667, 'medium': 0.407, 'hard': 0.308, 'extra': 0.5, 'all': 0.42}
Previously observed: original 48%, diagnostic replay 52% (different protocol)


In [14]:
from datetime import datetime,timezone
REPORT_FILE=RESULTS_DIR/'spider_1_5b_official_subset_evaluation.json'
report={
    'project':'SQLSmith','milestone':'A','notebook':'03_Spider_Official_Evaluation',
    'model':'Qwen2.5-Coder-1.5B-Instruct','source':'saved predictions from Notebook 01',
    'dataset':'Spider 1.0 development set','subset':'first 50 questions',
    'sample_size':len(df),'metric':'official test-suite execution accuracy (subset)',
    'official_subset_accuracy':score,'scores_by_difficulty':score_by_difficulty,
    'original_provisional_accuracy':0.48,'diagnostic_replay_accuracy':0.52,
    'plug_value':False,'keep_distinct':False,
    'evaluator_repo':'https://github.com/taoyds/test-suite-sql-eval',
    'evaluator_commit':evaluator_commit,
    'python_version':sys.version.split()[0],
    'baseline_source_sha256':source_hash,
    'gold_file_sha256':sha256_file(GOLD_FILE),
    'pred_file_sha256':sha256_file(PRED_FILE),
    'test_suite_db_counts':suite_counts,
    'test_suite_archive_sha256':sha256_file(TEST_SUITE_ARCHIVE),
    'full_dev_evaluated':False,
    'generated_at_utc':datetime.now(timezone.utc).isoformat()
}
RESULTS_DIR.mkdir(parents=True,exist_ok=True)
REPORT_FILE.write_text(json.dumps(report,indent=2),encoding='utf-8')
assert sha256_file(BASELINE_FILE)==source_hash, 'Source baseline was changed during evaluation'
for p in (GOLD_FILE,PRED_FILE,LOG_FILE,REPORT_FILE):
    assert p.exists() and p.stat().st_size>0, f'Missing or empty artifact: {p}'
print(json.dumps(report,indent=2))
print('\nPASS: Saved all artifacts; original baseline unchanged')

{
  "project": "SQLSmith",
  "milestone": "A",
  "notebook": "03_Spider_Official_Evaluation",
  "model": "Qwen2.5-Coder-1.5B-Instruct",
  "source": "saved predictions from Notebook 01",
  "dataset": "Spider 1.0 development set",
  "subset": "first 50 questions",
  "sample_size": 50,
  "metric": "official test-suite execution accuracy (subset)",
  "official_subset_accuracy": 0.42,
  "scores_by_difficulty": {
    "easy": 0.667,
    "medium": 0.407,
    "hard": 0.308,
    "extra": 0.5,
    "all": 0.42
  },
  "original_provisional_accuracy": 0.48,
  "diagnostic_replay_accuracy": 0.52,
  "plug_value": false,
  "keep_distinct": false,
  "evaluator_repo": "https://github.com/taoyds/test-suite-sql-eval",
  "evaluator_commit": "e97acc546ecbee8fa27fa8dbf025ef61493a876c",
  "python_version": "3.13.18",
  "baseline_source_sha256": "3d714667a99969fb16090359564da301d1b478eeba2bc1c376760d96fc953f3e",
  "gold_file_sha256": "2523871ac4fa312076a17448302d1778c254e5b072bb0e3cc509dd79e7eeb930",
  "pred_fil